In [4]:
import requests
from bs4 import BeautifulSoup
import sqlite3
import time
import re

# 定数
DB_NAME = "google_repos.db"
TARGET_URL = "https://github.com/google?tab=repositories"

def setup_database():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS repositories (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            language TEXT,
            stars TEXT
        )
    ''')
    cursor.execute('DELETE FROM repositories')
    conn.commit()
    conn.close()
    print(f"[INFO] データベース {DB_NAME} を初期化しました。")

def get_google_repos():
    print(f"[INFO] {TARGET_URL} からデータを取得中...")
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
    
    try:
        response = requests.get(TARGET_URL, headers=headers)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"[ERROR] ページの取得に失敗しました: {e}")
        return []

    soup = BeautifulSoup(response.text, 'html.parser')

    data_list = []
    seen_repos = set() # 重複防止用

    # 【最終手段】ページ内の「すべてのリンク(aタグ)」を取得し、URLのパターンでリポジトリを探す
    # 正規表現: /google/任意の文字 (ただしスラッシュ等は含まない)
    repo_pattern = re.compile(r'^/google/([^/?&]+)$')
    
    all_links = soup.find_all('a', href=True)
    print(f"[DEBUG] ページ内の全リンク数: {len(all_links)}")
    
    for a in all_links:
        href = a['href']
        match = repo_pattern.match(href)
        
        if match:
            repo_name = match.group(1)
            
            # リポジトリではないシステム的なリンクを除外
            if repo_name in ['repositories', 'people', 'projects', 'packages', 'sponsors', 'settings', 'discussions']:
                continue
            
            # すでに処理したリポジトリならスキップ
            if repo_name in seen_repos:
                continue
                
            # リポジトリ名として追加
            seen_repos.add(repo_name)
            
            # ここから親要素を辿って、セットになっている言語やスター数を探す
            # aタグの親の親...と辿って、行(li)やコンテナ(div)を見つける
            container = a.find_parent('li') 
            if not container:
                # liで囲まれていない場合、divの可能性もあるので3階層くらい上まで探してみる
                container = a.find_parent('div', class_='col-10') or a.parent.parent
            
            # データ取得の初期値
            language = "No Language"
            stars = "0"
            
            if container:
                # 1. 言語
                lang_tag = container.find('span', itemprop='programmingLanguage')
                if lang_tag:
                    language = lang_tag.get_text(strip=True)
                else:
                    # itempropがない場合、色付きの丸いアイコンの次にあるテキストを探す等の処理が必要だが
                    # 簡易的にspanタグの中身をチェック
                    mr3 = container.find('span', class_='mr-3')
                    if mr3:
                         language = mr3.get_text(strip=True)

                # 2. スター数 (/stargazers へのリンクを探す)
                star_tag = container.find('a', href=re.compile(r'/stargazers$'))
                if star_tag:
                    stars = star_tag.get_text(strip=True).replace(',', '')

            repo_data = (repo_name, language, stars)
            data_list.append(repo_data)
            
            print(f"[SCRAPED] {repo_name} | {language} | ⭐ {stars}")
            
            time.sleep(1)
            
            if len(data_list) >= 30:
                break
    
    if not data_list:
        print("[ERROR] 全リンク検索でもデータが見つかりませんでした。Bot対策が強力な可能性があります。")

    return data_list

def save_to_db(data_list):
    if not data_list:
        print("[WARN] 保存するデータがありません。")
        return

    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.executemany('INSERT INTO repositories (name, language, stars) VALUES (?, ?, ?)', data_list)
    conn.commit()
    conn.close()
    print(f"[INFO] {len(data_list)} 件のデータを保存しました。")

def show_saved_data():
    print("\n--- データベース保存結果 (SELECT文による表示) ---")
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute('SELECT * FROM repositories')
    rows = cursor.fetchall()
    
    if not rows:
        print("データが存在しません。")
    else:
        print(f"{'ID':<5} | {'Name':<35} | {'Language':<15} | {'Stars':<10}")
        print("-" * 75)
        for row in rows:
            print(f"{row[0]:<5} | {row[1]:<35} | {row[2]:<15} | {row[3]:<10}")
        
    conn.close()

if __name__ == "__main__":
    setup_database()
    scraped_data = get_google_repos()
    save_to_db(scraped_data)
    show_saved_data()

[INFO] データベース google_repos.db を初期化しました。
[INFO] https://github.com/google?tab=repositories からデータを取得中...
[DEBUG] ページ内の全リンク数: 203
[SCRAPED] material-design-icons | No Language | ⭐ 52.6k
[SCRAPED] guava | Java | ⭐ 51.3k
[SCRAPED] zx | JavaScript | ⭐ 44.9k
[SCRAPED] styleguide | HTML | ⭐ 38.7k
[SCRAPED] leveldb | C++ | ⭐ 38.4k
[SCRAPED] googletest | C++ | ⭐ 37.5k
[SCRAPED] brotli | TypeScript | ⭐ 14445
[SCRAPED] cddlconv | TypeScript | ⭐ 11
[SCRAPED] jaxite | Python | ⭐ 88
[SCRAPED] nearby | C++ | ⭐ 887
[SCRAPED] orbax | Python | ⭐ 455
[SCRAPED] nomulus | Java | ⭐ 1767
[SCRAPED] device-infra | Java | ⭐ 58
[SCRAPED] gemma.cpp | C++ | ⭐ 6619
[SCRAPED] xls | C++ | ⭐ 1375
[SCRAPED] site-kit-wp | JavaScript | ⭐ 1337
[INFO] 16 件のデータを保存しました。

--- データベース保存結果 (SELECT文による表示) ---
ID    | Name                                | Language        | Stars     
---------------------------------------------------------------------------
17    | material-design-icons               | No Language     | 52.6k     